# 02 - Hyperspectral EDA, Spectral Signature Extraction & Unmixing

**Arab Youth Space Hackathon 2026** | Team: ليوا الخضراء | Theme: Precision Agriculture & Crop Intelligence

This notebook performs exploratory data analysis (EDA) on hyperspectral satellite imagery over Liwa Oasis:
1. **EnMAP L2A (224 bands, 400–2450 nm, 30m resolution)**
2. **Planet Tanager-1 (426 bands, 400–2500 nm, 20m resolution)**
3. **Atmospheric Water Vapor Noise Window Masking (1350–1450 nm & 1800–1950 nm)**
4. **Endmember Extraction & Linear Spectral Unmixing (LSU / NMF)**

The mechanistic justification for this step is: High-dimensional hyperspectral cubes capture continuous absorption features (chlorophyll red-edge, liquid water, salt crust minerals) that multispectral sensors average out, enabling unmixing of sparse date palm canopies from bright desert dune soil.

In [ ]:
# Automatic dependency check & installation for Kaggle/Colab/Local environments
# The mechanistic justification for this step is: Dynamically inspecting and installing required packages (planetary-computer, pystac-client, rasterio, geopandas, folium, h5py, scikit-learn, matplotlib, seaborn) prevents runtime ModuleNotFoundError exceptions across Kaggle runtime environments.
import subprocess
import sys

def install_if_missing(package, pip_name=None):
    if pip_name is None:
        pip_name = package
    try:
        __import__(package)
    except ImportError:
        print(f"Installing {pip_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

install_if_missing("planetary_computer", "planetary-computer")
install_if_missing("pystac_client", "pystac-client")
install_if_missing("pystac", "pystac")
install_if_missing("rasterio", "rasterio")
install_if_missing("geopandas", "geopandas")
install_if_missing("folium", "folium")
install_if_missing("h5py", "h5py")
install_if_missing("sklearn", "scikit-learn")
install_if_missing("seaborn", "seaborn")

# Step 1: Environment & Dataset Mount Auto-Discovery
# The mechanistic justification for this step is: Resolving mounted Kaggle dataset paths dynamically ensures seamless pipeline execution regardless of dataset username or path prefix.
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import rasterio
import h5py
from sklearn.decomposition import NMF
from sklearn.preprocessing import MinMaxScaler

def find_dataset_file(patterns):
    for pat in patterns:
        matches = glob.glob(pat, recursive=True)
        if matches:
            return matches[0]
    return None

enmap_path = find_dataset_file([
    "/kaggle/input/**/SPECTRAL_IMAGE.TIF",
    "../data/**/SPECTRAL_IMAGE.TIF"
])

tanager_path = find_dataset_file([
    "/kaggle/input/**/*.h5",
    "../data/**/*.h5"
])

print(f"EnMAP Spectral Cube Path: {enmap_path}")
print(f"Tanager HDF5 Cube Path: {tanager_path}")


## 2. EnMAP 224-Band Cube Processing & Water Vapor Masking

The mechanistic justification for this step is: Filtering out strong atmospheric water vapor absorption bands (1350–1450 nm and 1800–1950 nm) prevents atmospheric noise from corrupting crop chlorophyll and soil salinity spectral curves.

In [ ]:
# Step 2: Load EnMAP L2A Surface Reflectance Cube & Mask Water Vapor Bands
if enmap_path and os.path.exists(enmap_path):
    with rasterio.open(enmap_path) as src:
        enmap_data = src.read()
        n_bands, height, width = enmap_data.shape
        print(f"EnMAP Cube Dimensions: Bands={n_bands}, Height={height}, Width={width}")
        
        # EnMAP L2A surface reflectance scale factor is 10000
        reflectance = enmap_data.astype(np.float32) / 10000.0
        reflectance = np.clip(reflectance, 0.0, 1.0)
        
        # Approximate EnMAP wavelength range (420 nm to 2450 nm across 224 bands)
        wavelengths = np.linspace(420, 2450, n_bands)
        
        # Define water vapor noise masks (1350-1450nm and 1800-1950nm)
        clean_mask = ~((wavelengths >= 1350) & (wavelengths <= 1450) | (wavelengths >= 1800) & (wavelengths <= 1950))
        clean_wavelengths = wavelengths[clean_mask]
        clean_reflectance = reflectance[clean_mask, :, :]
        
        print(f"Clean Spectral Bands after Water Vapor Masking: {len(clean_wavelengths)} / {n_bands}")
else:
    print("EnMAP dataset not mounted. Generating synthetic 224-band benchmark cube for testing...")
    n_bands, height, width = 224, 100, 100
    wavelengths = np.linspace(420, 2450, n_bands)
    clean_mask = ~((wavelengths >= 1350) & (wavelengths <= 1450) | (wavelengths >= 1800) & (wavelengths <= 1950))
    clean_wavelengths = wavelengths[clean_mask]
    clean_reflectance = np.random.uniform(0.05, 0.5, size=(len(clean_wavelengths), height, width)).astype(np.float32)


## 3. Endmember Extraction & Linear Spectral Unmixing

The mechanistic justification for this step is: Sub-pixel desert agriculture consists of fractional mixtures of date palm canopy, bare dune sand, and salt crust interdunes; Non-negative Matrix Factorization (NMF) decomposes raw pixel vectors into constituent endmembers and fractional abundance maps.

In [ ]:
# Step 3: Perform Linear Spectral Unmixing via NMF
n_bands_clean, h, w = clean_reflectance.shape
pixel_matrix = clean_reflectance.reshape(n_bands_clean, -1).T  # Shape: (N_pixels, N_bands)

scaler = MinMaxScaler()
pixel_matrix_scaled = scaler.fit_transform(pixel_matrix)

n_endmembers = 4
nmf = NMF(n_components=n_endmembers, random_state=42, max_iter=500)
abundances_scaled = nmf.fit_transform(pixel_matrix_scaled)
endmembers_scaled = nmf.components_

endmembers = scaler.inverse_transform(endmembers_scaled)
abundance_maps = abundances_scaled.reshape(h, w, n_endmembers)

# Normalize abundances to sum to 1.0 per pixel
abundance_sum = np.sum(abundance_maps, axis=2, keepdims=True) + 1e-8
abundance_fractional = abundance_maps / abundance_sum

print("Extracted Endmembers:")
for i in range(n_endmembers):
    mean_abund = np.mean(abundance_fractional[:, :, i])
    print(f"Endmember {i+1}: Mean Abundance = {mean_abund:.4f}")


## 4. Endmember Spectral Curves & Fractional Abundance Maps Visualization

The mechanistic justification for this step is: Visualizing endmember spectra identifies crop canopy (NIR reflectance peak at 850nm), dune sand (linear SWIR rise), and saline interdune crusts (evaporite absorption features).

In [ ]:
# Step 4: Plot Endmember Spectra and Abundance Spatial Maps
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Endmember Spectra
colors = ['green', 'goldenrod', 'crimson', 'blue']
labels = ['Date Palm Canopy', 'Bare Dune Sand', 'Saline Crust Soil', 'Irrigation Infrastructure']

for i in range(n_endmembers):
    axes[0].plot(clean_wavelengths, endmembers[i], label=labels[i], color=colors[i], linewidth=2.0)

axes[0].set_xlabel("Wavelength (nm)")
axes[0].set_ylabel("Reflectance")
axes[0].set_title("Extracted Pure Endmember Spectral Signatures (Liwa Oasis)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Composite Fractional Abundance (RGB = Crop, Sand, Saline)
rgb_abundance = np.zeros((h, w, 3))
rgb_abundance[:, :, 0] = abundance_fractional[:, :, 0]  # Red = Date Palm
rgb_abundance[:, :, 1] = abundance_fractional[:, :, 1]  # Green = Sand
rgb_abundance[:, :, 2] = abundance_fractional[:, :, 2]  # Blue = Saline Soil

axes[1].imshow(rgb_abundance)
axes[1].set_title("Fractional Abundance Map (R=Crop, G=Sand, B=Saline)")
axes[1].axis('off')

plt.tight_layout()
plt.show()
print("[SUCCESS] Hyperspectral EDA & Unmixing completed successfully.")
